# Support Vector Machines

We wil implement both hard-margin SVMs and soft-margin SVMs from scratch on a toy dataset. Apart from `NumPy`, we would need to take the help of `SciPy` for solving the quadratic programming problem.

## Hard-Margin SVM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [12, 12]

In [ ]:
#### DATA: DO NOT EDIT THIS CELL ####
X = np.array([[1, -3], [1, 0], [4, 1], [3, 7], [0, -2],
             [-1, -6], [2, 5], [1, 2], [0, -1], [-1, -4],
             [0, 7], [1, 5], [-4, 4], [2, 9], [-2, 2],
             [-2, 0], [-3, -2], [-2, -4], [3, 10], [-3, -8]]).T
y = np.array([1, 1, 1, 1, 1,
             1, 1, 1, 1, 1,
             -1, -1, -1, -1, -1,
             -1, -1, -1, -1, -1])

### Problem-1

$\mathbf{X}$ is a data-matrix of shape $(d, n)$. $\mathbf{y}$ is a vector of labels of size $(n, )$. What is the value of $n$ and $d$?

In [ ]:
print("d: ", X.shape[0] , ", n: " , X.shape[1] )

### Problem-2

Visualize the dataset given to you using a scatter plot. Colour points which belong to class $+1$ $\color{green}{\text{green}}$ and those that belong to $-1$ $\color{red}{\text{red}}$. Inspect the data visually and determine its linear separability.

In [ ]:
plt.figure()

plt.scatter(X[0 , y==1] , X[1 , y==1] , color = "green" , label="Class +1" , edgecolors = "black" , s = 100)

plt.scatter(X[0 , y==-1] , X[1 , y==-1] , color = "black" , label="Class +1" , edgecolors = "white" , s = 100)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

plt.title("Dataset Visualization")

plt.legend()

plt.grid(True)

plt.show()

### Problem-3

Compute the object $\mathbf{Y}$ that appears in the dual problem. What kind of an object is $\mathbf{Y}$?

### Solution

## Problem 3: Computing the Matrix $\mathbf{Y}$ for the SVM Dual Problem

---

### What is SVM trying to do?

SVM (Support Vector Machine) tries to find a **hyperplane**:

$$\mathbf{w}^T \mathbf{x} + b = 0$$

that separates the two classes with **maximum margin**.

- $\mathbf{w}$ = weight vector (direction of the hyperplane)
- $b$ = bias (position of the hyperplane)

The margin is given by $\dfrac{2}{||\mathbf{w}||}$, so maximizing the margin means **minimizing** $||\mathbf{w}||$.

---

### Primal Problem

$$\min_{\mathbf{w}, b} \frac{1}{2} ||\mathbf{w}||^2$$

$$\text{subject to: } y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 \quad \forall i$$

---

### Why Dual Problem?

We convert the primal to its **Lagrangian dual** form. This is useful because:
1. The solution depends only on **dot products** $\mathbf{x}_i^T \mathbf{x}_j$
2. Only **support vectors** (points near the boundary) matter
3. It enables the **kernel trick** for non-linear boundaries

The Lagrangian is:

$$L(\mathbf{w}, b, \boldsymbol{\alpha}) = \frac{1}{2}||\mathbf{w}||^2 - \sum_{i=1}^{n} \alpha_i \left[ y_i(\mathbf{w}^T \mathbf{x}_i + b) - 1 \right]$$

where $\alpha_i \geq 0$ are **Lagrange multipliers** (one per training point).

---

### KKT Conditions (setting derivatives to zero)

**With respect to $\mathbf{w}$:**

$$\frac{\partial L}{\partial \mathbf{w}} = 0 \implies \mathbf{w} = \sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i$$

**With respect to $b$:**

$$\frac{\partial L}{\partial b} = 0 \implies \sum_{i=1}^{n} \alpha_i y_i = 0$$

---

### Final Dual Problem

Substituting the KKT conditions back:

$$\max_{\boldsymbol{\alpha}} \sum_{i=1}^{n} \alpha_i - \frac{1}{2} \sum_{i=1}^{n} \sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j \mathbf{x}_i^T \mathbf{x}_j$$

$$\text{subject to: } \alpha_i \geq 0, \quad \sum_{i=1}^{n} \alpha_i y_i = 0$$

---

### Defining $\mathbf{Y}$

The double summation term can be written compactly as $\dfrac{1}{2}\boldsymbol{\alpha}^T \mathbf{Y} \boldsymbol{\alpha}$, where:

$$Y_{ij} = y_i \, y_j \, \mathbf{x}_i^T \mathbf{x}_j$$

Written in matrix form:

$$\boxed{\mathbf{Y} = (\mathbf{y}\mathbf{y}^T) \odot (X^T X)}$$

where:
- $\mathbf{y}\mathbf{y}^T$ is the **outer product** of the label vector with itself (shape: $n \times n$), with $(i,j)$ entry $= y_i y_j$
- $X^T X$ is the **Gram matrix** (shape: $n \times n$), with $(i,j)$ entry $= \mathbf{x}_i^T \mathbf{x}_j$
- $\odot$ denotes **element-wise (Hadamard) multiplication**

---

### Properties of $\mathbf{Y}$

| Property | Value |
|---|---|
| Shape | $n \times n = 20 \times 20$ |
| Type | Square Matrix |
| Symmetry | Symmetric ($Y_{ij} = Y_{ji}$) |
| Nature | Positive Semi-Definite (PSD) |

$\mathbf{Y}$ is **symmetric** because $y_i y_j \mathbf{x}_i^T \mathbf{x}_j = y_j y_i \mathbf{x}_j^T \mathbf{x}_i$.

In [ ]:
# y ko column vector banao (20,1) shape
y_col = y.reshape(-1, 1)   # shape: (20, 1)

# outer product: y_col @ y_col.T → (20,20) matrix
# har (i,j) entry = yi * yj
yyt = y_col @ y_col.T      # shape: (20, 20)

# X.T @ X → (20,20) matrix  
# har (i,j) entry = xi . xj (dot product)
# X shape hai (2,20), toh X.T shape (20,2)
# (20,2) @ (2,20) = (20,20) ✅
XtX = X.T @ X              # shape: (20, 20)

# Element-wise multiply karo dono ko
# Y[i,j] = yi*yj * xi.xj
Y = yyt * XtX              # shape: (20, 20)

print("Shape of Y:", Y.shape)   # (20, 20)
print("Type of object: 2D Square Matrix")

# Verify symmetry
print("Is Y symmetric?", np.allclose(Y, Y.T))  # True hona chahiye

### Problem-4

Let $\boldsymbol{\alpha}$ be the dual variable. The dual objective is of the form:

$$
f(\boldsymbol{\alpha}) = \boldsymbol{\alpha}^T \mathbf{1} - \cfrac{1}{2} \cdot \boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha}
$$



Compute the matrix $\mathbf{Q}$ for this problem and find the sum of its elements. What properties does the matrix $\mathbf{Q}$ have? What is the nature of the objective function?


### Solution

## Problem 4: Computing Matrix $\mathbf{Q}$ and Properties of the Dual Objective

---

### Step 1: Connection Between Problem 3 and Problem 4

In **Problem 3**, we computed matrix $\mathbf{Y}$ where:

$$Y_{ij} = y_i \, y_j \, \mathbf{x}_i^T \mathbf{x}_j$$

In **Problem 4**, the dual objective is given as:

$$f(\boldsymbol{\alpha}) = \boldsymbol{\alpha}^T \mathbf{1} - \frac{1}{2} \cdot \boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha}$$

Our original dual problem (derived in Problem 3) was:

$$\max_{\boldsymbol{\alpha}} \sum_{i=1}^{n} \alpha_i - \frac{1}{2} \sum_{i=1}^{n} \sum_{j=1}^{n} \alpha_i \alpha_j \, y_i y_j \, \mathbf{x}_i^T \mathbf{x}_j$$

Now compare the two term by term:

| Term in $f(\boldsymbol{\alpha})$ | Corresponding term in dual |
|---|---|
| $\boldsymbol{\alpha}^T \mathbf{1}$ | $\sum_i \alpha_i$ (sum of all $\alpha_i$) |
| $\frac{1}{2} \boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha}$ | $\frac{1}{2} \sum_i \sum_j \alpha_i \alpha_j y_i y_j \mathbf{x}_i^T \mathbf{x}_j$ |

By direct comparison:

$$\boxed{Q_{ij} = y_i \, y_j \, \mathbf{x}_i^T \mathbf{x}_j}$$

**Conclusion: $\mathbf{Q}$ and $\mathbf{Y}$ are the same matrix!**

$$\mathbf{Q} = \mathbf{Y} = (\mathbf{y}\mathbf{y}^T) \odot (X^T X)$$

where $\odot$ denotes element-wise (Hadamard) multiplication.

---

### Step 2: Properties of $\mathbf{Q}$

#### Property 1: Symmetric

$$Q_{ij} = y_i y_j \mathbf{x}_i^T \mathbf{x}_j = y_j y_i \mathbf{x}_j^T \mathbf{x}_i = Q_{ji}$$

So $\mathbf{Q} = \mathbf{Q}^T$ — the matrix is **symmetric**.

#### Property 2: Positive Semi-Definite (PSD)

For any vector $\mathbf{v} \in \mathbb{R}^n$:

$$\mathbf{v}^T \mathbf{Q} \mathbf{v} \geq 0$$

**Why?** $\mathbf{Q}$ is essentially a **Gram matrix** (matrix of dot products) scaled by label products. Gram matrices are always PSD because they represent inner products in some space.

We can verify this by checking that all eigenvalues of $\mathbf{Q}$ are $\geq 0$.

---

### Step 3: Nature of the Objective Function $f$

$$f(\boldsymbol{\alpha}) = \underbrace{\boldsymbol{\alpha}^T \mathbf{1}}_{\text{linear (neither convex nor concave)}} - \underbrace{\frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha}}_{\text{quadratic term}}$$

The **Hessian** (matrix of second derivatives) of $f$ is:

$$\nabla^2 f(\boldsymbol{\alpha}) = -\mathbf{Q}$$

Now:
- $\mathbf{Q}$ is **Positive Semi-Definite** $\implies$ $-\mathbf{Q}$ is **Negative Semi-Definite**
- Hessian is Negative Semi-Definite $\implies$ $f$ is **Concave**

**Intuition:** The $-\frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha}$ term with the **minus sign** flips the quadratic bowl upside down — giving a mountain shape (concave) instead of a valley shape (convex).

$$\text{Convex} \rightarrow \text{valley shape (minimum exists)} \quad \text{Concave} \rightarrow \text{mountain shape (maximum exists)}$$

Since $f$ is **concave**, maximizing it is a well-posed problem — there exists a unique global maximum.

---

### Step 4: Summary of Properties

| Property | Value |
|---|---|
| Shape of $\mathbf{Q}$ | $n \times n = 20 \times 20$ |
| Symmetric? | Yes, $\mathbf{Q} = \mathbf{Q}^T$ |
| Definiteness | Positive Semi-Definite (PSD) |
| Sum of all elements | $976$ |
| Nature of $f(\boldsymbol{\alpha})$ | Concave |

---

### Code

```python
# Step 1: Build Q matrix (same as Y from Problem 3)
y_col = y.reshape(-1, 1)       # shape: (20, 1) — column vector
yyt   = y_col @ y_col.T        # shape: (20, 20) — outer product, entry (i,j) = yi * yj
XtX   = X.T @ X                # shape: (20, 20) — Gram matrix, entry (i,j) = xi . xj
Q     = yyt * XtX              # element-wise multiply — Q[i,j] = yi*yj * xi.xj

# Step 2: Sum of all elements of Q
sum_Q = np.sum(Q)
print("Sum of elements of Q:", sum_Q)   # Output: 976

# Step 3: Check symmetry
print("Is Q symmetric?", np.allclose(Q, Q.T))   # True

# Step 4: Check PSD — all eigenvalues must be >= 0
eigenvalues = np.linalg.eigvalsh(Q)
print("Minimum eigenvalue:", np.min(eigenvalues))
print("Is Q PSD?", np.all(eigenvalues >= -1e-10))   # True (small tolerance for floating point)
```


In [ ]:
# Step 1: Y/Q matrix banao (Problem 3 se same)
y_col = y.reshape(-1, 1)      # shape: (20, 1)
yyt   = y_col @ y_col.T       # shape: (20, 20) → yi*yj har entry pe
XtX   = X.T @ X               # shape: (20, 20) → xi.xj har entry pe
Q     = yyt * XtX             # element-wise multiply → Q[i,j] = yi*yj*xi.xj

# Step 2: Sum of elements
sum_Q = np.sum(Q)
print("Sum of elements of Q:", sum_Q)

# Step 3: Symmetry check
print("Is Q symmetric?", np.allclose(Q, Q.T))

# Step 4: PSD check → sabhi eigenvalues >= 0 honi chahiye
eigenvalues = np.linalg.eigvalsh(Q)
print("Min eigenvalue:", np.min(eigenvalues))
print("Is Q PSD?", np.all(eigenvalues >= -1e-10))  # small tolerance for floating point

### Problem-5

Since `SciPy`'s optimization routines take the form of minimizing a function, we will recast $f$ as follows:

$$
f(\boldsymbol{\alpha}) =  \cfrac{1}{2} \cdot \boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha} - \boldsymbol{\alpha}^T \mathbf{1}
$$

We now have to solve :

$$
\min \limits_{\boldsymbol{\alpha} \geq 0} \quad f(\boldsymbol{\alpha})
$$

Note that $\max$ changes to $\min$ since we changed the sign of the objective function.

<hr>

Write a function `loss` that returns the value of objective function $f(\boldsymbol{\alpha})$ for argument $\boldsymbol{\alpha}$. Compute the value of `loss` at $\boldsymbol{\alpha} = \mathbf{1}$.

**Note**: The reason for naming the function `loss` is that we will be using `SciPy`'s `scipy.optimize.minize` routine.


### Solution

## Problem 5: Loss Function and SciPy Minimization

---

### Step 1: Why Did We Change the Sign?

Our original dual problem was:

$$\max_{\boldsymbol{\alpha}} \quad f(\boldsymbol{\alpha}) = \boldsymbol{\alpha}^T \mathbf{1} - \frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha}$$

**SciPy's `minimize` can only minimize — it cannot maximize directly.**

We use a simple mathematical trick:

$$\max \; f(\boldsymbol{\alpha}) \equiv \min \; -f(\boldsymbol{\alpha})$$

*"Maximizing a function is the same as minimizing its negative."*

So our new objective (which we now minimize) is:

$$f(\boldsymbol{\alpha}) = \frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha} - \boldsymbol{\alpha}^T \mathbf{1}$$

---

### Step 2: Understanding Each Term

$$f(\boldsymbol{\alpha}) = \underbrace{\frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha}}_{\text{Term 1}} - \underbrace{\boldsymbol{\alpha}^T \mathbf{1}}_{\text{Term 2}}$$

**Term 1:** $\frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha}$

- $\boldsymbol{\alpha}$ has shape $(n,)$, $\mathbf{Q}$ has shape $(n \times n)$
- $\mathbf{Q}\boldsymbol{\alpha} \rightarrow (n \times n)(n,) = (n,)$
- $\boldsymbol{\alpha}^T(\mathbf{Q}\boldsymbol{\alpha}) \rightarrow (n,) \cdot (n,) =$ **scalar**

**Term 2:** $\boldsymbol{\alpha}^T \mathbf{1}$

- $\mathbf{1}$ is a vector of all ones, shape $(n,)$
- $\boldsymbol{\alpha}^T \mathbf{1} = \sum_{i=1}^{n} \alpha_i$ — just the **sum of all $\alpha_i$**

So $f(\boldsymbol{\alpha})$ returns a **single scalar value**. ✅

---

### Step 3: Computing $f(\boldsymbol{\alpha})$ at $\boldsymbol{\alpha} = \mathbf{1}$

When $\boldsymbol{\alpha} = \mathbf{1}$ (all elements equal to 1):

**Term 1:**

$$\frac{1}{2}\mathbf{1}^T \mathbf{Q} \mathbf{1} = \frac{1}{2} \sum_{i=1}^{n}\sum_{j=1}^{n} Q_{ij} = \frac{1}{2} \times 976 = 488$$

(Because $\mathbf{1}^T \mathbf{Q} \mathbf{1}$ = sum of all elements of $\mathbf{Q}$ = 976, from Problem 4)

**Term 2:**

$$\mathbf{1}^T \mathbf{1} = \sum_{i=1}^{20} 1 = 20$$

**Final value:**

$$f(\mathbf{1}) = 488 - 20 = \boxed{468}$$

---

### Step 4: What is `scipy.optimize.minimize`?

`scipy.optimize.minimize` is a function that finds the **minimum of any given function**.

Basic usage:
```python
from scipy.optimize import minimize
result = minimize(fun, x0, ...)
```

- `fun` → the function to minimize (our `loss` function)
- `x0` → starting point (initial guess for $\boldsymbol{\alpha}$)

**Important:** The `loss` function must accept exactly one argument — the $\boldsymbol{\alpha}$ vector. SciPy passes this automatically. We use `Q` as a global variable inside the function.

---

### Code

```python
from scipy.optimize import minimize

def loss(alpha):
    # Term 1: 0.5 * alpha^T @ Q @ alpha
    # Q @ alpha  →  (20,20) @ (20,) = (20,)
    # alpha @ result  →  scalar
    term1 = 0.5 * alpha @ Q @ alpha

    # Term 2: alpha^T @ 1 = sum of all alphas
    # np.ones(len(alpha)) creates [1, 1, ..., 1] of shape (20,)
    term2 = alpha @ np.ones(len(alpha))

    return term1 - term2

# Test: alpha = 1 vector (all ones)
alpha_test = np.ones(20)          # shape: (20,)
print("loss at alpha=1:", loss(alpha_test))   # Output: 468.0
```

---

### Summary

| Quantity | Value |
|---|---|
| Term 1: $\frac{1}{2}\mathbf{1}^T \mathbf{Q} \mathbf{1}$ | $\frac{976}{2} = 488$ |
| Term 2: $\mathbf{1}^T \mathbf{1}$ | $20$ |
| $f(\mathbf{1})$ = `loss(np.ones(20))` | $\mathbf{468}$ |

In [ ]:
from scipy.optimize import minimize

def loss(alpha):
    # alpha: shape (20,) — ye SciPy dega automatically
    
    # Term 1: 0.5 * alpha^T @ Q @ alpha
    # Q @ alpha → (20,20) @ (20,) = (20,)
    # alpha @ (Q @ alpha) → scalar
    term1 = 0.5 * alpha @ Q @ alpha
    
    # Term 2: alpha^T @ 1 = sum of all alphas
    # np.ones(len(alpha)) → [1,1,1,...,1] shape (20,)
    term2 = alpha @ np.ones(len(alpha))
    
    return term1 - term2

# alpha = 1 vector banao (saare 1s)
alpha_test = np.ones(20)   # shape: (20,)

# loss compute karo
print("loss at alpha=1:", loss(alpha_test))   # Output: 468.0

### Problem-6

Write a function named `jac` that computes the gradient, $\nabla f(\boldsymbol{\alpha})$, given $\boldsymbol{\alpha}$ as argument. Compute the value of `jac` at $\boldsymbol{\alpha} = \mathbf{1}$ and print the sum of the components of the gradient vector.

**Note**: `jac` stands for Jacobian. In our case, we don't have a vector valued function. So, this will just be the gradient.

### Solution

## Problem 6: Gradient of the Objective Function (Jacobian)

---

### Step 1: Gradient Kya Hota Hai?

Socho ek simple example — agar $f(x) = x^2$ hai, toh:

$$\frac{df}{dx} = 2x$$

Ye derivative batata hai — *"x thoda badhaao toh f kitna badlega?"*

Ab hamare paas **ek number nahi, ek vector** $\boldsymbol{\alpha}$ hai jisme 20 elements hain:

$$\boldsymbol{\alpha} = [\alpha_1, \alpha_2, \ldots, \alpha_{20}]$$

Toh **gradient** = har ek $\alpha_i$ ke saath $f$ ka partial derivative:

$$\nabla f(\boldsymbol{\alpha}) = \begin{bmatrix} \frac{\partial f}{\partial \alpha_1} \\ \frac{\partial f}{\partial \alpha_2} \\ \vdots \\ \frac{\partial f}{\partial \alpha_{20}} \end{bmatrix}$$

Ye bhi ek **vector** hai — shape $(20,)$. ✅

---

### Step 2: Gradient Calculate Karo

Haara function hai:

$$f(\boldsymbol{\alpha}) = \frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha} - \boldsymbol{\alpha}^T \mathbf{1}$$

Iske **do terms** hain — dono ka alag alag derivative nikalte hain.

---

#### Term 1 ka Gradient: $\frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha}$

Ye **quadratic form** hai. Iska ek standard rule hota hai:

$$\frac{\partial}{\partial \boldsymbol{\alpha}} \left( \frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha} \right) = \mathbf{Q}\boldsymbol{\alpha}$$

**Kyun?** Simple analogy se samjho:

| Scalar case | Vector case |
|---|---|
| $\frac{d}{dx}\left(\frac{1}{2}q x^2\right) = qx$ | $\frac{\partial}{\partial \boldsymbol{\alpha}}\left(\frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha}\right) = \mathbf{Q}\boldsymbol{\alpha}$ |

($\mathbf{Q}$ symmetric hai isliye $\frac{1}{2}(\mathbf{Q} + \mathbf{Q}^T)\boldsymbol{\alpha} = \mathbf{Q}\boldsymbol{\alpha}$)

---

#### Term 2 ka Gradient: $\boldsymbol{\alpha}^T \mathbf{1}$

Ye **linear term** hai. Iska standard rule:

$$\frac{\partial}{\partial \boldsymbol{\alpha}} \left( \boldsymbol{\alpha}^T \mathbf{1} \right) = \mathbf{1}$$

**Kyun?** Simple analogy:

| Scalar case | Vector case |
|---|---|
| $\frac{d}{dx}(cx) = c$ | $\frac{\partial}{\partial \boldsymbol{\alpha}}(\boldsymbol{\alpha}^T \mathbf{1}) = \mathbf{1}$ |

---

#### Final Gradient (dono terms combine karo):

$$\boxed{\nabla f(\boldsymbol{\alpha}) = \mathbf{Q}\boldsymbol{\alpha} - \mathbf{1}}$$

---

### Step 3: Gradient at $\boldsymbol{\alpha} = \mathbf{1}$

$$\nabla f(\mathbf{1}) = \mathbf{Q}\mathbf{1} - \mathbf{1}$$

**$\mathbf{Q}\mathbf{1}$ kya hai?**

Jab hum kisi matrix ko ones ke vector se multiply karte hain, toh result mein $i$-th element = $\mathbf{Q}$ ki $i$-th row ka sum:

$$(\mathbf{Q}\mathbf{1})_i = \sum_{j=1}^{20} Q_{ij}$$

Phir usme se $\mathbf{1}$ (saare ones ka vector) subtract karte hain.

**Sum of all gradient components:**

$$\sum_{i=1}^{20} (\nabla f)_i = \sum_{i=1}^{20}\sum_{j=1}^{20} Q_{ij} - \sum_{i=1}^{20} 1 = 976 - 20 = \mathbf{956}$$

(976 = sum of all elements of $\mathbf{Q}$, jo Problem 4 mein nikala tha)

---

### Step 4: Jacobian Kya Hota Hai?

**Jacobian** generally vector-valued functions ke liye hota hai — jab output bhi ek vector ho. Agar $\mathbf{f}: \mathbb{R}^n \rightarrow \mathbb{R}^m$ ho toh:

$$J = \begin{bmatrix} \frac{\partial f_1}{\partial \alpha_1} & \cdots & \frac{\partial f_1}{\partial \alpha_n} \\ \vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial \alpha_1} & \cdots & \frac{\partial f_m}{\partial \alpha_n} \end{bmatrix}$$

Haare case mein $f$ sirf **ek scalar** return karta hai ($m = 1$), toh Jacobian = gradient vector hi ban jaata hai!

**SciPy mein `jac` kyun dete hain?**

SciPy ka `minimize` internally gradient use karta hai taaki samjhe — *"kis direction mein jaane se function aur chota hoga?"*

- Agar `jac` nahi dete → SciPy numerically estimate karta hai → **slow aur less accurate**
- Agar `jac` dete hain (analytically) → **fast aur accurate** ✅

---

### Code

```python
def jac(alpha):
    # Gradient formula: Q @ alpha - 1
    
    # Q @ alpha  →  (20,20) @ (20,) = (20,) vector
    # np.ones(len(alpha))  →  [1, 1, ..., 1] shape (20,)
    # Result: (20,) vector — ek gradient value har alpha_i ke liye
    
    return Q @ alpha - np.ones(len(alpha))

# Test at alpha = 1
alpha_test = np.ones(20)
gradient = jac(alpha_test)

print("Gradient at alpha=1:", gradient)
print("Sum of gradient components:", np.sum(gradient))   # Output: 956.0
```

---

### Summary

| Quantity | Value |
|---|---|
| Formula for $\nabla f(\boldsymbol{\alpha})$ | $\mathbf{Q}\boldsymbol{\alpha} - \mathbf{1}$ |
| $\mathbf{Q}\mathbf{1}$ | Row-wise sum of $\mathbf{Q}$ = vector of shape $(20,)$ |
| Sum of gradient at $\boldsymbol{\alpha} = \mathbf{1}$ | $976 - 20 = \mathbf{956}$ |

In [ ]:
def jac(alpha):
    # Gradient formula: Q @ alpha - 1
    
    # Q @ alpha → (20,20) @ (20,) = (20,) vector
    # np.ones(len(alpha)) → [1,1,...,1] shape (20,)
    # Result: (20,) vector — ek gradient value har alpha_i ke liye
    
    return Q @ alpha - np.ones(len(alpha))

# Test at alpha = 1
alpha_test = np.ones(20)
gradient = jac(alpha_test)

print("Gradient at alpha=1:", gradient)
print("Sum of gradient components:", np.sum(gradient))  # Output: 956.0

### Problem-7

Finally, we have most of the ingredients to solve the dual problem:

$$
\min \limits_{\boldsymbol{\alpha} \geq 0} \quad \cfrac{1}{2} \cdot \boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha} - \boldsymbol{\alpha}^T \mathbf{1}
$$

Go through this [document](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize) to understand how `scipy.optimize.minimize` works. Few pointers:

(1)  You should pass five arguments to `scipy.optimize.minimize`: `loss`, `jac`, `alpha_init`, `method`, `Bounds`

(2) Use the method `SLSQP`. You can treat this as a black-box.

(3) Set the initial value of `alpha_init` to zero.

(4) Use `scipy.optimize.Bounds` to trigger the $\boldsymbol{\alpha} \geq 0$ constraint.

Compute the sum of components of the optimal solution, $\boldsymbol{\alpha}^*$. Enter the nearest integer as your answer.


### Solution

## Problem 7: Solving the Dual Problem using SciPy

---

### Step 1: What Are We Doing Here?

So far we have built all the ingredients:

| What | Symbol | Built in |
|---|---|---|
| Matrix $\mathbf{Q}$ | $Q_{ij} = y_i y_j \mathbf{x}_i^T \mathbf{x}_j$ | Problem 3, 4 |
| Loss function | $f(\boldsymbol{\alpha}) = \frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q}\boldsymbol{\alpha} - \boldsymbol{\alpha}^T \mathbf{1}$ | Problem 5 |
| Gradient function | $\nabla f(\boldsymbol{\alpha}) = \mathbf{Q}\boldsymbol{\alpha} - \mathbf{1}$ | Problem 6 |

Now we hand everything to **SciPy** and it will automatically find the best $\boldsymbol{\alpha}^*$ that minimizes `loss`.

---

### Step 2: What is `scipy.optimize.minimize`?

Think of it as a **robot**. You tell it:
- *"This is the function to minimize"* → `loss`
- *"This is the gradient"* → `jac`
- *"Start from here"* → `alpha_init`
- *"Use this algorithm"* → `SLSQP`
- *"These are the constraints"* → `Bounds`

The robot walks around and finds the minimum on its own. ✅

---

### Step 3: Understanding Each Argument

#### Argument 1: `fun = loss`

The function we want to minimize:

$$f(\boldsymbol{\alpha}) = \frac{1}{2}\boldsymbol{\alpha}^T \mathbf{Q} \boldsymbol{\alpha} - \boldsymbol{\alpha}^T \mathbf{1}$$

---

#### Argument 2: `jac`

The gradient function we wrote:

$$\nabla f(\boldsymbol{\alpha}) = \mathbf{Q}\boldsymbol{\alpha} - \mathbf{1}$$

SciPy uses this to decide direction — *"which way should I move to reduce the loss faster?"*

---

#### Argument 3: `x0 = alpha_init`

The starting point — *"where should the robot begin?"*

We start from the zero vector:

$$\boldsymbol{\alpha}_{init} = \mathbf{0} = [0, 0, \ldots, 0]$$

**Why zero?** Because our constraint is $\boldsymbol{\alpha} \geq 0$, and zero is a valid starting point that satisfies this constraint.

---

#### Argument 4: `method = 'SLSQP'`

**SLSQP** = Sequential Least Squares Programming

This is an optimization algorithm that:
- Can handle constraints (like $\boldsymbol{\alpha} \geq 0$)
- Is gradient-based (that is why we pass `jac`)
- Is efficient for quadratic problems like ours

Treat this as a **black box** — we do not need to know the internals, just know that it minimizes with constraints. ✅

---

#### Argument 5: `bounds = Bounds(lb, ub)`

Our constraint is: $\boldsymbol{\alpha} \geq 0$

This means each $\alpha_i$:
- **Lower bound** $= 0$ (cannot go below zero)
- **Upper bound** $= +\infty$ (no upper limit)

`scipy.optimize.Bounds(lb, ub)` sets this for all 20 components at once:

$$0 \leq \alpha_i < +\infty \quad \forall \, i$$

---

### Step 4: Reading the Result

`minimize` returns a **result object**. The important fields are:

| Field | Meaning |
|---|---|
| `result.x` | Optimal $\boldsymbol{\alpha}^*$ vector, shape $(20,)$ |
| `result.fun` | Minimum value of `loss` at $\boldsymbol{\alpha}^*$ |
| `result.success` | `True` if optimization converged successfully |

---

### Step 5: What to Expect — SVM Intuition

In SVM, only the **support vectors** (points closest to the boundary) have non-zero $\alpha_i$. All other points get $\alpha_i = 0$.

This means most of the 20 values in $\boldsymbol{\alpha}^*$ will be **zero** — only a few will be positive. The sum of components of $\boldsymbol{\alpha}^*$ will therefore be a small number.

---

### Code

```python
from scipy.optimize import minimize, Bounds

# Step 1: Initial value — zero vector
# Why zero? It is a valid starting point satisfying alpha >= 0
alpha_init = np.zeros(20)        # shape: (20,)

# Step 2: Set bounds — alpha >= 0 for all components
# Bounds(lb, ub) → lb = lower bound = 0, ub = upper bound = infinity
bounds = Bounds(lb=0, ub=np.inf)

# Step 3: Call minimize
result = minimize(
    fun=loss,          # function to minimize
    x0=alpha_init,     # starting point
    jac=jac,           # gradient function
    method='SLSQP',    # optimization algorithm (treat as black box)
    bounds=bounds      # constraint: alpha >= 0
)

# Step 4: Extract optimal alpha*
alpha_star = result.x
print("Optimal alpha*:", alpha_star)
print("Sum of components of alpha*:", np.sum(alpha_star))
print("Optimization successful?", result.success)
```

---

### Summary

| Quantity | Value |
|---|---|
| Starting point $\boldsymbol{\alpha}_{init}$ | $\mathbf{0}$ (zero vector) |
| Constraint | $\boldsymbol{\alpha} \geq 0$ via `Bounds(lb=0, ub=np.inf)` |
| Algorithm | `SLSQP` (handles constraints + uses gradient) |
| Result | `result.x` gives optimal $\boldsymbol{\alpha}^*$ |
| Answer | `np.sum(alpha_star)` rounded to nearest integer |

In [ ]:
from scipy.optimize import minimize, Bounds

# Step 1: Initial value of alpha — zero vector
alpha_init = np.zeros(20)   # shape: (20,) — saare zeros
                             # kyun zero? valid starting point hai alpha >= 0 ke liye

# Step 2: Bounds set karo — alpha >= 0
# Bounds(lb, ub) → lb = lower bound, ub = upper bound
# lb = 0 → har alpha_i >= 0
# ub = np.inf → koi upper limit nahi
bounds = Bounds(lb=0, ub=np.inf)

# Step 3: minimize call karo
result = minimize(
    fun=loss,          # jo function minimize karna hai
    x0=alpha_init,     # starting point
    jac=jac,           # gradient function
    method='SLSQP',    # optimization algorithm
    bounds=bounds      # alpha >= 0 constraint
)

# Step 4: Optimal alpha* nikalo
alpha_star = result.x
print("Optimal alpha*:", alpha_star)
print("Sum of components of alpha*:", np.sum(alpha_star))
print("Optimization successful?", result.success)

### Problem-8

Find all the support vectors. Print the indices (zero-indexing) in the data-matrix where these support vectors are found.

### Solution

## Problem 8: Finding the Support Vectors

---

### Step 1: Support Vector Kya Hota Hai?

SVM ka goal ek **hyperplane** dhundna hai jo dono classes ko maximum margin se alag kare.

**Support Vectors** = wo points jo margin ki **boundary pe** hote hain — hyperplane ke sabse paas wale points.

Mathematically, support vectors wo points hain jahan:

$$y_i(\mathbf{w}^T \mathbf{x}_i + b) = 1$$

Baaki saare points ke liye ye value $> 1$ hoti hai — matlab wo margin ke bahar hote hain aur decision boundary se koi lena dena nahi unka.

---

### Step 2: $\alpha_i$ aur Support Vectors Ka Relation (KKT Conditions)

Dual problem solve karne ke baad humein $\boldsymbol{\alpha}^*$ milta hai. **KKT (Karush-Kuhn-Tucker) Conditions** se ek bahut important rule aata hai:

$$\alpha_i \cdot \left[ y_i(\mathbf{w}^T \mathbf{x}_i + b) - 1 \right] = 0 \quad \forall \, i$$

Is condition ka matlab:

| Situation | $\alpha_i$ | Point ka type |
|---|---|---|
| Point margin ke **bahar** hai | $\alpha_i = 0$ | Not a support vector |
| Point margin ki **boundary pe** hai | $\alpha_i > 0$ | **Support Vector** ✅ |

So the rule is simply:

$$\boxed{\alpha_i > 0 \iff \text{point } i \text{ is a Support Vector}}$$

---

### Step 3: Threshold Kyun Use Karte Hain?

Computers **floating point numbers** use karte hain. Kabhi kabhi koi $\alpha_i$ technically zero hona chahiye but computer `0.000000001` jaisi value deta hai due to numerical errors.

Isliye hum check karte hain:

$$\alpha_i > \text{threshold} \quad (\text{e.g., } 10^{-5})$$

Zero ke bahut paas ki values ko hum zero maante hain — this is called **numerical tolerance**.

---

### Step 4: Intuition — Kitne Support Vectors Honge?

In SVM, only the support vectors **define the decision boundary**. All other points can be removed and the boundary would remain the same.

- Most $\alpha_i = 0$ → those points are **far from the boundary** → not support vectors
- Few $\alpha_i > 0$ → those points are **on the margin** → **support vectors**

In this 2D dataset, we expect only **3 to 5 support vectors** out of 20 total points.

---

### Code

```python
# alpha_star was obtained from Problem 7 (optimal solution from minimize)

# Support vectors are indices where alpha_i > threshold
# threshold handles floating point errors near zero
threshold = 1e-5   # 0.00001

# np.where returns indices where condition is True
# [0] is used because np.where returns a tuple
support_vector_indices = np.where(alpha_star > threshold)[0]

print("Support Vector Indices:", support_vector_indices)
print("Number of Support Vectors:", len(support_vector_indices))
print("Alpha values at support vectors:", alpha_star[support_vector_indices])
```

---

### Summary

| Concept | Detail |
|---|---|
| Support Vector condition | $\alpha_i > 0$ (from KKT conditions) |
| Why threshold? | Floating point errors — use $\alpha_i > 10^{-5}$ |
| How to find indices? | `np.where(alpha_star > 1e-5)[0]` |
| Expected count | 3 to 5 support vectors out of 20 points |

In [ ]:
# alpha_star Problem 7 mein mila tha (optimal solution)

# Support vectors wo indices hain jahan alpha_i > threshold
threshold = 1e-5   # 0.00001 — floating point errors se bachne ke liye

# np.where returns indices jahan condition True ho
support_vector_indices = np.where(alpha_star > threshold)[0]

print("Support Vector Indices:", support_vector_indices)
print("Number of Support Vectors:", len(support_vector_indices))
print("Alpha values at support vectors:", alpha_star[support_vector_indices])

### Problem-9

Find the optimal weight vector $\mathbf{w}^*$. Round each component of the optimal weight vector to the nearest integer.

### Solution

## Problem 9: Finding the Optimal Weight Vector $\mathbf{w}^*$

---

### Step 1: What is $\mathbf{w}^*$?

Recall the SVM hyperplane:

$$\mathbf{w}^T \mathbf{x} + b = 0$$

Here $\mathbf{w}$ is the **weight vector** — it defines the direction of the hyperplane (decision boundary).

We have already found $\boldsymbol{\alpha}^*$ in Problem 7. Now we use it to recover $\mathbf{w}^*$.

---

### Step 2: Formula for $\mathbf{w}^*$ — Where Does It Come From?

In Problem 3, we derived the KKT condition by setting $\frac{\partial L}{\partial \mathbf{w}} = 0$:

$$\mathbf{w} = \sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i$$

This means $\mathbf{w}^*$ is a **weighted sum of all training points**, where the weight of point $i$ is $\alpha_i^* y_i$.

But remember — **only support vectors have $\alpha_i^* > 0$**, all others are zero!

So in practice only the support vectors contribute:

$$\mathbf{w}^* = \sum_{i \in \text{support vectors}} \alpha_i^* \, y_i \, \mathbf{x}_i$$

---

### Step 3: Writing It in Matrix Form

Instead of looping over each point, we use matrix multiplication.

We have:
- $X$ shape: $(2, 20)$ — 2 features, 20 points
- $\boldsymbol{\alpha}^*$ shape: $(20,)$
- $\mathbf{y}$ shape: $(20,)$

**Step 3a:** Compute $\alpha_i^* \cdot y_i$ for each point (element-wise multiply):

$$\boldsymbol{\alpha}^* \odot \mathbf{y} \quad \text{shape: } (20,)$$

**Step 3b:** Multiply $X$ with this vector:

$$\mathbf{w}^* = X \cdot (\boldsymbol{\alpha}^* \odot \mathbf{y})$$

$$\text{shape: } (2, 20) \times (20,) = (2,)$$

So $\mathbf{w}^*$ is a **2-dimensional vector** (because our data has 2 features). ✅

---

### Step 4: What Does Each Component Mean?

$$w^*_1 = \sum_{i=1}^{20} \alpha_i^* \, y_i \, x_{i,1} \quad \text{(contribution along feature 1)}$$

$$w^*_2 = \sum_{i=1}^{20} \alpha_i^* \, y_i \, x_{i,2} \quad \text{(contribution along feature 2)}$$

Since all non-support-vector $\alpha_i = 0$, only the few support vectors actually contribute to these sums.

---

### Code

```python
# Formula: w* = X @ (alpha_star * y)

# alpha_star * y  →  element-wise multiply, shape (20,)
#                    each entry = alpha_i * y_i
# X @ result      →  (2, 20) @ (20,) = (2,)
#                    w*[0] = sum over all i of alpha_i * y_i * x_i1
#                    w*[1] = sum over all i of alpha_i * y_i * x_i2

w_star = X @ (alpha_star * y)

print("Optimal weight vector w*:", w_star)
print("Rounded w*:", np.round(w_star))
```

---

### Summary

| Quantity | Detail |
|---|---|
| Formula | $\mathbf{w}^* = \sum_i \alpha_i^* y_i \mathbf{x}_i = X(\boldsymbol{\alpha}^* \odot \mathbf{y})$ |
| Shape of $\mathbf{w}^*$ | $(2,)$ — one value per feature |
| Who contributes? | Only support vectors (all others have $\alpha_i = 0$) |
| Final answer | `np.round(w_star)` gives the rounded components |

In [ ]:
# alpha_star — Problem 7 se mila (shape: (20,))
# y — labels (shape: (20,))
# X — data matrix (shape: (2, 20))

# Step 1: alpha_i * yi — element wise multiply
# alpha_star * y → (20,) * (20,) = (20,)
alpha_y = alpha_star * y

# Step 2: X @ alpha_y
# (2, 20) @ (20,) = (2,) → w* vector
w_star = X @ alpha_y

print("Optimal w*:", w_star)
print("Rounded w*:", np.round(w_star))

### Problem-10

Plot the decision boundary along with the supporting hyperplanes. Note where the support vectors lie in this plot. How many red points lie on the supporting hyperplanes? How many green points lie on the supporting hyperplanes?

### Solution

## Problem 10: Plotting the Decision Boundary and Supporting Hyperplanes

---

### Step 1: Teen Lines Kya Hain?

SVM mein teen important lines hoti hain:

| Line | Equation | Matlab |
|---|---|---|
| Decision Boundary | $\mathbf{w}^T\mathbf{x} + b = 0$ | Beech wali line — dono classes ko alag karti hai |
| $+1$ Hyperplane | $\mathbf{w}^T\mathbf{x} + b = +1$ | Green support vectors yahan hote hain |
| $-1$ Hyperplane | $\mathbf{w}^T\mathbf{x} + b = -1$ | Red support vectors yahan hote hain |

The region between the $+1$ and $-1$ hyperplanes is the **margin** — SVM maximizes this gap.

---

### Step 2: $b^*$ Kaise Nikalte Hain?

We need the bias $b^*$ to plot the lines. For any support vector $i$, the following holds exactly:

$$y_i(\mathbf{w}^{*T}\mathbf{x}_i + b^*) = 1$$

Rearranging:

$$b^* = y_i - \mathbf{w}^{*T}\mathbf{x}_i$$

In practice, we take the **average over all support vectors** for numerical stability:

$$b^* = \frac{1}{|\text{SVs}|} \sum_{i \in \text{SVs}} \left( y_i - \mathbf{w}^{*T}\mathbf{x}_i \right)$$

---

### Step 3: Lines Ko 2D Mein Kaise Plot Karen?

In 2D, a hyperplane is just a **line**. The general equation is:

$$w_1 x_1 + w_2 x_2 + b = c$$

Solving for $x_2$ (so we can plot it):

$$x_2 = \frac{c - b - w_1 x_1}{w_2}$$

For the three lines, we use $c = 0, +1, -1$:

$$x_2^{\text{boundary}} = \frac{0 - b^* - w_1^* x_1}{w_2^*} \quad \text{(decision boundary)}$$

$$x_2^{+1} = \frac{1 - b^* - w_1^* x_1}{w_2^*} \quad \text{(+1 supporting hyperplane)}$$

$$x_2^{-1} = \frac{-1 - b^* - w_1^* x_1}{w_2^*} \quad \text{(-1 supporting hyperplane)}$$

---

### Step 4: Support Vectors Aur Hyperplanes Ka Relation

By definition of SVM (from KKT conditions):

- Green support vectors lie **exactly on** the $+1$ hyperplane
- Red support vectors lie **exactly on** the $-1$ hyperplane

So to count how many green/red points lie on supporting hyperplanes, we simply count the support vectors of each class.

---

### Code

```python
# Step 1: Compute b* — average over all support vectors
# For each support vector i: b = y_i - w* . x_i
# w_star @ X[:, support_vector_indices]  →  dot product for each support vector
b_star = np.mean(y[support_vector_indices] - w_star @ X[:, support_vector_indices])
print("Optimal bias b*:", b_star)

# Step 2: x1 range for plotting
x1_range = np.linspace(X[0].min() - 1, X[0].max() + 1, 100)

# Step 3: Compute x2 for each of the three lines
# x2 = (c - b* - w1*x1) / w2
x2_decision = ( 0 - b_star - w_star[0] * x1_range) / w_star[1]  # c = 0
x2_pos      = ( 1 - b_star - w_star[0] * x1_range) / w_star[1]  # c = +1
x2_neg      = (-1 - b_star - w_star[0] * x1_range) / w_star[1]  # c = -1

# Step 4: Plot
plt.figure()

# All data points
plt.scatter(X[0, y ==  1], X[1, y ==  1], color='green',
            label='Class +1', s=100, edgecolors='black', zorder=3)
plt.scatter(X[0, y == -1], X[1, y == -1], color='red',
            label='Class -1', s=100, edgecolors='black', zorder=3)

# Highlight support vectors with a blue circle
plt.scatter(X[0, support_vector_indices], X[1, support_vector_indices],
            s=300, facecolors='none', edgecolors='blue',
            linewidths=2, label='Support Vectors', zorder=4)

# Three lines
plt.plot(x1_range, x2_decision, 'k-',  linewidth=2,   label='Decision Boundary (w.x + b = 0)')
plt.plot(x1_range, x2_pos,      'g--', linewidth=1.5,  label='+1 Hyperplane (w.x + b = +1)')
plt.plot(x1_range, x2_neg,      'r--', linewidth=1.5,  label='-1 Hyperplane (w.x + b = -1)')

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('SVM: Decision Boundary and Supporting Hyperplanes')
plt.legend()
plt.grid(True)
plt.show()

# Step 5: Count support vectors per class
sv_green = np.sum(y[support_vector_indices] == 1)
sv_red   = np.sum(y[support_vector_indices] == -1)
print("Green points on +1 hyperplane:", sv_green)
print("Red points on -1 hyperplane:",   sv_red)
```

---

### Summary

| Quantity | Detail |
|---|---|
| $b^*$ formula | Average of $y_i - \mathbf{w}^{*T}\mathbf{x}_i$ over all support vectors |
| Decision boundary | $\mathbf{w}^T\mathbf{x} + b = 0$ → black solid line |
| $+1$ hyperplane | $\mathbf{w}^T\mathbf{x} + b = +1$ → green dashed line |
| $-1$ hyperplane | $\mathbf{w}^T\mathbf{x} + b = -1$ → red dashed line |
| Green SVs on $+1$ hyperplane | Count from `y[support_vector_indices] == 1` |
| Red SVs on $-1$ hyperplane | Count from `y[support_vector_indices] == -1` |

In [ ]:
# Step 1: b* nikalo — average over all support vectors
b_star = np.mean(y[support_vector_indices] - w_star @ X[:, support_vector_indices])
print("Optimal bias b*:", b_star)

# Step 2: x1 ka range banao plot ke liye
x1_range = np.linspace(X[0].min() - 1, X[0].max() + 1, 100)

# Step 3: Teen lines ke liye x2 calculate karo
# Line equation: w1*x1 + w2*x2 + b = c
# => x2 = (c - b - w1*x1) / w2
x2_decision  = (-b_star - w_star[0] * x1_range) / w_star[1]  # c = 0
x2_pos       = (1 - b_star - w_star[0] * x1_range) / w_star[1]  # c = +1
x2_neg       = (-1 - b_star - w_star[0] * x1_range) / w_star[1]  # c = -1

# Step 4: Plot banao
plt.figure()

# Data points
plt.scatter(X[0, y ==  1], X[1, y ==  1], color='green',
            label='Class +1', s=100, edgecolors='black', zorder=3)
plt.scatter(X[0, y == -1], X[1, y == -1], color='red',
            label='Class -1', s=100, edgecolors='black', zorder=3)

# Support vectors ko highlight karo (bada circle)
plt.scatter(X[0, support_vector_indices], X[1, support_vector_indices],
            s=300, facecolors='none', edgecolors='blue',
            linewidths=2, label='Support Vectors', zorder=4)

# Teen lines plot karo
plt.plot(x1_range, x2_decision, 'k-',  linewidth=2, label='Decision Boundary')
plt.plot(x1_range, x2_pos,      'g--', linewidth=1.5, label='+1 Hyperplane')
plt.plot(x1_range, x2_neg,      'r--', linewidth=1.5, label='-1 Hyperplane')

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('SVM Decision Boundary and Support Hyperplanes')
plt.legend()
plt.grid(True)
plt.show()

## Soft-margin SVM

We now turn to soft-margin SVMs. Adapt the hard-margin code that you have written for the soft-margin problem. The only change you have to make is to introduce an upper bound for $\boldsymbol{\alpha}$, which is the hyperparameter $C$.


## Work

## Soft-Margin SVM: From Hard-Margin to Soft-Margin

---

### Step 1: Hard-Margin SVM Ki Limitation

Hard-margin SVM only works when data is **perfectly linearly separable**.

In real world, if even one point is on the wrong side or inside the margin, hard-margin SVM **fails completely** — no solution exists.

---

### Step 2: Soft-Margin — Allowing Some Mistakes

Soft-margin SVM introduces a **slack variable** $\xi_i \geq 0$ for each point:

$$y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1 - \xi_i$$

| Value of $\xi_i$ | Meaning |
|---|---|
| $\xi_i = 0$ | Point correctly classified, outside the margin ✅ |
| $0 < \xi_i \leq 1$ | Point inside the margin, but on correct side |
| $\xi_i > 1$ | Point misclassified — on the wrong side ❌ |

The soft-margin primal problem becomes:

$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2}||\mathbf{w}||^2 + C \sum_{i=1}^{n} \xi_i$$

$$\text{subject to: } y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0 \quad \forall \, i$$

---

### Step 3: What Changes in the Dual Problem?

**Hard-margin dual constraint:**

$$0 \leq \alpha_i$$

**Soft-margin dual constraint:**

$$\boxed{0 \leq \alpha_i \leq C}$$

That is the **only change**! Everything else — $\mathbf{Q}$, `loss`, `jac` — remains exactly the same.

The upper bound $C$ comes from the slack variables in the primal. Introducing slack adds a new constraint which translates to an upper bound on $\alpha_i$ in the dual.

---

### Step 4: What is $C$?

$C$ is a **hyperparameter** that controls the trade-off between margin width and misclassification:

| Value of $C$ | Behaviour |
|---|---|
| $C \to \infty$ | No mistakes allowed → behaves like hard-margin SVM |
| $C$ very small | Many mistakes allowed → wider margin, more misclassifications |
| $C$ medium | Balance between margin width and classification accuracy |

**Intuition:**
- Large $C$ → strict → every point must be correctly classified
- Small $C$ → lenient → some misclassifications are acceptable in exchange for a wider margin

---

### Step 5: Code Change — Only One Line!

**Hard-margin:**
```python
bounds = Bounds(lb=0, ub=np.inf)   # 0 <= alpha_i <= infinity
```

**Soft-margin:**
```python
bounds = Bounds(lb=0, ub=C)        # 0 <= alpha_i <= C
```

Everything else — `Q`, `loss`, `jac`, `minimize` call — stays exactly the same. ✅

---

### Summary

| Aspect | Hard-Margin | Soft-Margin |
|---|---|---|
| Constraint on $\alpha_i$ | $0 \leq \alpha_i \leq \infty$ | $0 \leq \alpha_i \leq C$ |
| Allows misclassification? | No | Yes (controlled by $C$) |
| Works on non-separable data? | No | Yes |
| Code change needed | — | Only `ub=np.inf` → `ub=C` in `Bounds` |

### Problem-11

Plot the decision boundary and the supporting hyperplane for the following values of $C$.

(1) $C = 0.01$

(2) $C = 0.1$

(3) $C = 1$

(4) $C = 10$

Plot all of them in a $2 \times 2$ subplot. Study the tradeoff between the following quantities:

(1) Width of the margin.

(2) Number of points that lie within the margin or on the wrong side. This is often called **margin violation**.


### Solution

## Problem 11: Soft-SVM Decision Boundary for Different Values of $C$

---

### Step 1: What Does $C$ Control?

In soft-margin SVM, the dual constraint is:

$$0 \leq \alpha_i \leq C$$

The margin width is given by:

$$\text{Margin} = \frac{2}{||\mathbf{w}^*||}$$

$C$ controls the trade-off between margin width and margin violations:

| $C$ | $\alpha_i$ range | $||\mathbf{w}^*||$ | Margin Width | Violations |
|---|---|---|---|---|
| $0.01$ | $[0,\ 0.01]$ | Small | **Largest** | Most |
| $0.1$ | $[0,\ 0.1]$ | Medium-small | Large | Many |
| $1$ | $[0,\ 1]$ | Medium | Medium | Few |
| $10$ | $[0,\ 10]$ | Large | **Smallest** | Fewest |

**Intuition:**
- Small $C$ → lenient → wide margin, many points allowed inside or on wrong side
- Large $C$ → strict → tight margin, behaves like hard-margin SVM

---

### Step 2: What is a Margin Violation?

A **margin violation** is any point that lies inside the margin or on the wrong side:

$$y_i(\mathbf{w}^{*T}\mathbf{x}_i + b^*) < 1 \implies \text{margin violation}$$

This includes:
- Points inside the margin but on correct side: $0 < y_i(\mathbf{w}^T\mathbf{x}_i + b) < 1$
- Points on the wrong side: $y_i(\mathbf{w}^T\mathbf{x}_i + b) < 0$

---

### Step 3: Only One Line Changes from Hard-Margin

**Hard-margin:**
```python
bounds = Bounds(lb=0, ub=np.inf)
```

**Soft-margin:**
```python
bounds = Bounds(lb=0, ub=C)
```

Everything else — `Q`, `loss`, `jac`, `minimize` call — stays exactly the same.

---

### Code

```python
from scipy.optimize import minimize, Bounds

def solve_soft_svm(C):
    """Solve soft-margin SVM for a given value of C"""
    alpha_init = np.zeros(20)

    # Only change from hard-margin: upper bound is C instead of infinity
    bounds = Bounds(lb=0, ub=C)

    result = minimize(
        fun=loss,
        x0=alpha_init,
        jac=jac,
        method='SLSQP',
        bounds=bounds
    )

    alpha_star = result.x
    w_star     = X @ (alpha_star * y)

    # b* — average over all support vectors
    threshold = 1e-5
    sv_idx    = np.where(alpha_star > threshold)[0]
    b_star    = np.mean(y[sv_idx] - w_star @ X[:, sv_idx])

    return alpha_star, w_star, b_star, sv_idx


def plot_svm(ax, w_star, b_star, sv_idx, C, alpha_star):
    """Plot decision boundary and supporting hyperplanes on a given axis"""

    # Data points
    ax.scatter(X[0, y ==  1], X[1, y ==  1],
               color='green', s=80, edgecolors='black',
               zorder=3, label='Class +1')
    ax.scatter(X[0, y == -1], X[1, y == -1],
               color='red', s=80, edgecolors='black',
               zorder=3, label='Class -1')

    # Support vectors — blue circle around them
    ax.scatter(X[0, sv_idx], X[1, sv_idx],
               s=250, facecolors='none', edgecolors='blue',
               linewidths=2, zorder=4, label='Support Vectors')

    # Three lines: x2 = (c - b* - w1*x1) / w2
    x1_range    = np.linspace(X[0].min() - 1, X[0].max() + 1, 200)
    x2_boundary = ( 0 - b_star - w_star[0] * x1_range) / w_star[1]
    x2_pos      = ( 1 - b_star - w_star[0] * x1_range) / w_star[1]
    x2_neg      = (-1 - b_star - w_star[0] * x1_range) / w_star[1]

    ax.plot(x1_range, x2_boundary, 'k-',  linewidth=2,   label='Decision Boundary')
    ax.plot(x1_range, x2_pos,      'g--', linewidth=1.5,  label='+1 Hyperplane')
    ax.plot(x1_range, x2_neg,      'r--', linewidth=1.5,  label='-1 Hyperplane')

    # Count margin violations: y_i * (w*.x_i + b*) < 1
    scores     = y * (w_star @ X + b_star)
    violations = np.sum(scores < 1)
    margin     = 2 / np.linalg.norm(w_star)

    ax.set_title(f'C = {C} | Margin = {margin:.2f} | Violations = {violations}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(fontsize=7)
    ax.grid(True)


# 2x2 subplot for 4 values of C
C_values = [0.01, 0.1, 1, 10]

fig, axes = plt.subplots(2, 2)

for ax, C in zip(axes.flatten(), C_values):
    alpha_star, w_star, b_star, sv_idx = solve_soft_svm(C)
    plot_svm(ax, w_star, b_star, sv_idx, C, alpha_star)

plt.tight_layout()
plt.show()
```

---

### Tradeoff Summary

| $C$ | Margin Width | Margin Violations | Behaviour |
|---|---|---|---|
| $0.01$ | Largest | Most | Very lenient — wide margin, many violations |
| $0.1$ | Large | Many | Lenient |
| $1$ | Medium | Few | Balanced |
| $10$ | Smallest | Fewest | Strict — behaves like hard-margin SVM |

**Key insight:** As $C$ increases, the model becomes less tolerant of violations, the margin shrinks, and the solution approaches the hard-margin SVM.

In [ ]:
from scipy.optimize import minimize, Bounds

# Q, loss, jac same hain problem 5, 6 se

def solve_soft_svm(C):
    """Kisi bhi C ke liye soft-SVM solve kare"""
    alpha_init = np.zeros(20)
    
    # SIRF YE LINE BADLI HAI hard-margin se:
    # Hard: Bounds(lb=0, ub=np.inf)
    # Soft: Bounds(lb=0, ub=C)
    bounds = Bounds(lb=0, ub=C)
    
    result = minimize(
        fun=loss,
        x0=alpha_init,
        jac=jac,
        method='SLSQP',
        bounds=bounds
    )
    
    alpha_star = result.x
    w_star     = X @ (alpha_star * y)
    
    # b* — average over support vectors
    threshold = 1e-5
    sv_idx    = np.where(alpha_star > threshold)[0]
    b_star    = np.mean(y[sv_idx] - w_star @ X[:, sv_idx])
    
    return alpha_star, w_star, b_star, sv_idx


def plot_svm(ax, w_star, b_star, sv_idx, C, alpha_star):
    """Ek subplot mein decision boundary + hyperplanes plot kare"""
    
    # --- Data points ---
    ax.scatter(X[0, y ==  1], X[1, y ==  1],
               color='green', s=80, edgecolors='black',
               zorder=3, label='Class +1')
    ax.scatter(X[0, y == -1], X[1, y == -1],
               color='red', s=80, edgecolors='black',
               zorder=3, label='Class -1')
    
    # --- Support vectors (blue circle) ---
    ax.scatter(X[0, sv_idx], X[1, sv_idx],
               s=250, facecolors='none', edgecolors='blue',
               linewidths=2, zorder=4, label='Support Vectors')
    
    # --- Teen lines ---
    x1_range = np.linspace(X[0].min() - 1, X[0].max() + 1, 200)
    
    # x2 = (c - b* - w1*x1) / w2
    x2_boundary = ( 0 - b_star - w_star[0] * x1_range) / w_star[1]
    x2_pos      = ( 1 - b_star - w_star[0] * x1_range) / w_star[1]
    x2_neg      = (-1 - b_star - w_star[0] * x1_range) / w_star[1]
    
    ax.plot(x1_range, x2_boundary, 'k-',  linewidth=2,   label='Decision Boundary')
    ax.plot(x1_range, x2_pos,      'g--', linewidth=1.5,  label='+1 Hyperplane')
    ax.plot(x1_range, x2_neg,      'r--', linewidth=1.5,  label='-1 Hyperplane')
    
    # --- Margin violations count karo ---
    # y_i * (w*.x_i + b*) < 1 → violation
    scores     = y * (w_star @ X + b_star)
    violations = np.sum(scores < 1)
    margin     = 2 / np.linalg.norm(w_star)
    
    ax.set_title(f'C = {C} | Margin = {margin:.2f} | Violations = {violations}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(fontsize=7)
    ax.grid(True)


# --- 2x2 Subplot ---
C_values = [0.01, 0.1, 1, 10]

fig, axes = plt.subplots(2, 2)   # 2 rows, 2 columns

for ax, C in zip(axes.flatten(), C_values):
    # Har C ke liye SVM solve karo
    alpha_star, w_star, b_star, sv_idx = solve_soft_svm(C)
    # Plot karo
    plot_svm(ax, w_star, b_star, sv_idx, C, alpha_star)

plt.tight_layout()   # subplots overlap na karen
plt.show()